In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

ACTIVE_BACKEND = "API"   # "CPU" | "GPU" | "API"

PARQUET_PATH = Path(r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet")
TEXT_COLUMN = "full_text"
N_DOCS = 10  # None = alle Dokumente

In [ ]:
if ACTIVE_BACKEND == "CPU":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    model_path = hf_hub_download(
        repo_id="Qwen/Qwen3-Embedding-0.6B-GGUF",
        filename="Qwen3-Embedding-0.6B-Q8_0.gguf",
    )
    _llm = Llama(model_path=model_path, embedding=True, n_ctx=512, verbose=False)

    def embed_fn(texts):
        return np.vstack([
            np.array(_llm.create_embedding(t)["data"][0]["embedding"], dtype=np.float32)
            for t in tqdm(texts, desc="Embeddings (CPU)")
        ])

elif ACTIVE_BACKEND == "GPU":
    from sentence_transformers import SentenceTransformer

    _model = SentenceTransformer("google/embeddinggemma-300m")

    def embed_fn(texts):
        return _model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

elif ACTIVE_BACKEND == "API":
    from dotenv import load_dotenv
    from litellm import embedding as _litellm_embed

    load_dotenv()
    _API_MODEL = "openrouter/openai/text-embedding-3-large"

    def embed_fn(texts):
        return np.array([
            _litellm_embed(model=_API_MODEL, input=[t])["data"][0]["embedding"]
            for t in tqdm(texts, desc="Embeddings (API)")
        ], dtype=np.float32)

else:
    raise ValueError(f"Invalid ACTIVE_BACKEND: {ACTIVE_BACKEND!r}")

In [3]:
df = pd.read_parquet(PARQUET_PATH)
docs = (
    df[TEXT_COLUMN].dropna().astype(str).map(str.strip).loc[lambda s: s != ""]
    .head(N_DOCS).tolist() if N_DOCS else
    df[TEXT_COLUMN].dropna().astype(str).map(str.strip).loc[lambda s: s != ""].tolist()
)
print(f"{len(docs)} Dokumente geladen")

embeddings = embed_fn(docs)
print(f"Shape: {embeddings.shape}")
print(f"Erste 8 Werte: {embeddings[0][:8]}")

10 Dokumente geladen


Embeddings (API):   0%|          | 0/10 [00:00<?, ?it/s]

Shape: (10, 3072)
Erste 8 Werte: [-1.5517000e-02 -1.7697444e-02 -1.2231398e-02 -2.1729774e-03
 -7.7998033e-05  1.1350260e-02  6.7952215e-03  1.8832471e-02]
